<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/Analytics-Forecast/blob/main/Analytics_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

Data cleaning and Perparing

In [31]:
df = pd.read_csv("/content/data.csv")

In [45]:
df['Date'] = pd.to_datetime(df['Date'])

def generate_features(group):
    group['day_of_week'] = group['Date'].dt.dayofweek
    group['day_of_month'] = group['Date'].dt.day
    group['month'] = group['Date'].dt.month

    group['lag_1'] = group['SumofPrincipalOutstanding'].shift(1)
    group['lag_7'] = group['SumofPrincipalOutstanding'].shift(7)
    group['lag_30'] = group['SumofPrincipalOutstanding'].shift(30)
    group['rolling_mean_7'] = group['SumofPrincipalOutstanding'].shift(1).rolling(window=7).mean()

    return group

df_final = df.groupby('BranchID', group_keys=False).apply(generate_features)
df_final = df_final.dropna(subset=['lag_30', 'rolling_mean_7'])

/tmp/ipykernel_1960/3559314197.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final = df.groupby('BranchID', group_keys=False).apply(generate_features)


Defining the Features and Targets

In [47]:
features = ['BranchID', 'day_of_week', 'day_of_month', 'month',
            'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7']
target = 'SumofPrincipalOutstanding'

df_final['BranchID'] = df_final['BranchID'].astype('category')

split_date = df_final['Date'].max() - pd.Timedelta(days=30)
train = df_final[df_final['Date'] <= split_date]
valid = df_final[df_final['Date'] > split_date]

X_train, y_train = train[features], train[target]
X_valid, y_valid = valid[features], valid[target]

Light GBM dataset

In [49]:
dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=['BranchID'])
dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=['BranchID'], reference=dtrain)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42
}

model = lgb.train(params, dtrain, valid_sets=[dvalid], num_boost_round=1000)

In [ ]:
def generate_forecast(model, last_data, days_to_forecast):
    forecast_df = last_data.copy()

    for _ in range(days_to_forecast):
        last_rows = forecast_df.groupby('BranchID').tail(1).copy()
        last_rows['Date'] = last_rows['Date'] + pd.Timedelta(days=1)
        last_rows['day_of_week'] = last_rows['Date'].dt.dayofweek
        last_rows['day_of_month'] = last_rows['Date'].dt.day
        last_rows['month'] = last_rows['Date'].dt.month

        preds = model.predict(last_rows[features])
        last_rows['SumofPrincipalOutstanding'] = preds

        forecast_df = pd.concat([forecast_df, last_rows])

    return forecast_df

In [51]:
def predict_future_clean(model, df_recent, days_to_predict, features):
    future_list = []
    current_buffer = df_recent.copy()

    for i in range(days_to_predict):
        next_date = current_buffer['Date'].max() + pd.Timedelta(days=1)

        last_states = current_buffer.sort_values('Date').groupby('BranchID').tail(1).copy()

        tomorrow = last_states.copy()
        tomorrow['Date'] = next_date
        tomorrow['day_of_week'] = tomorrow['Date'].dt.dayofweek
        tomorrow['day_of_month'] = tomorrow['Date'].dt.day
        tomorrow['month'] = tomorrow['Date'].dt.month

        for branch in tomorrow['BranchID'].unique():
            history = current_buffer[current_buffer['BranchID'] == branch].sort_values('Date')

            tomorrow.loc[tomorrow['BranchID'] == branch, 'lag_1'] = history['SumofPrincipalOutstanding'].iloc[-1]
            tomorrow.loc[tomorrow['BranchID'] == branch, 'lag_7'] = history['SumofPrincipalOutstanding'].iloc[-7] if len(history) >= 7 else history['SumofPrincipalOutstanding'].iloc[-1]
            tomorrow.loc[tomorrow['BranchID'] == branch, 'rolling_mean_7'] = history['SumofPrincipalOutstanding'].tail(7).mean()

        preds = model.predict(tomorrow[features])
        tomorrow['predicted_value'] = preds

        tomorrow['SumofPrincipalOutstanding'] = preds

        future_list.append(tomorrow[['Date', 'BranchID', 'predicted_value']])
        current_buffer = pd.concat([current_buffer, tomorrow]).reset_index(drop=True)

    return pd.concat(future_list)

forecast_df = predict_future_clean(model, df_final, 7, features)

/tmp/ipykernel_1960/660444252.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  last_states = current_buffer.sort_values('Date').groupby('BranchID').tail(1).copy()
/tmp/ipykernel_1960/660444252.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  last_states = current_buffer.sort_values('Date').groupby('BranchID').tail(1).copy()
/tmp/ipykernel_1960/660444252.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  last_state

In [62]:
forecast_df = forecast_df.sort_values(['Date','BranchID'])
forecast_df.head(150)

,Date,BranchID,predicted_value
95688,2026-04-01,1,8754514.70
95689,2026-04-01,2,13664334.15
95690,2026-04-01,3,11114476.48
95691,2026-04-01,4,14284715.46
95692,2026-04-01,5,12456431.59
...,...,...,...
95833,2026-04-01,146,3688145.95
95834,2026-04-01,147,12410667.86
95835,2026-04-01,148,9583277.67
95836,2026-04-01,149,6863185.84
